# Python, memory and systems
## UCB MFE · Advanced Python I · October 19, 2026

This is a prepared teaching notebook. Work through the numbered examples from top to bottom, or run an individual example independently. Every example includes its own setup and imports. All market values are synthetic classroom inputs.

**Requirements:** Python 3.12+ and NumPy, already in the course environment. No API key, network call, database, GPU, model download, or external data file is required.

From the course repository, use `uv sync --locked` and choose that `.venv` as the notebook kernel. If your kernel already has NumPy, no installation is needed. Use **Restart Kernel and Run All** for a clean run. The saved outputs provide a fallback if the classroom environment is unavailable.

**Teaching rhythm:** ask the prediction question, run the cell, discuss the output, then open the answer. Instructor answers and teaching notes are in collapsible sections. Collapse them before projecting; notebook viewers differ in HTML support.

**Route:** D01–D03, D05–D10, D12–D16, D18–D20. D04, D11, D17 are optional. Timing and allocation figures vary by machine. Saved measurements are one example run, not performance guarantees.

The hardware illustrations and broader explanations are in the companion slides. Detailed GPU/KV-cache material belongs to Thursday, October 22.

# 1. Scope, references, and mutation

## D01 · Scope and hidden inputs

Core example · about 5 minutes · slide 6

**Predict before running:** What changes between the two calls to quoted_value? Does the function receive a new argument?

In [21]:
rate = 1.10

def quoted_value(amount):
    return amount * rate

print(round(quoted_value(100), 2))
rate = 1.20
print(round(quoted_value(100), 2))

def explicit_value(amount, fx_rate):
    return amount * fx_rate

print(round(explicit_value(100, 1.10), 2))
from math import isclose
assert isclose(explicit_value(100, 1.10), 110.0)

110.0
120.0


In [3]:
rate = 1.10

def quoted_value(amount):
    rate = 2.10
    return amount * rate

print(round(quoted_value(100), 2))
rate = 1.20
print(round(quoted_value(100), 2))
print(rate)

210.0
210.0
1.2


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** 110.0, 120.0, 110.0. The first function reads the current global rate when called. The explicit version makes that dependency visible. This is name lookup and hidden state, not random behavior.

**Teaching note:** Ask for all three outputs before running. Explain local, enclosing, global, built-in lookup. A notebook kernel keeps globals between cell executions. Restarting and running top-to-bottom helps expose accidental dependencies.

</details>

## D02 · Mutation, reassignment, identity

Core example · about 6 minutes · slide 7

**Predict before running:** After append, which names see 30? After b is reassigned, do a and b still refer to the same list?

In [13]:
a = [10, 20]
b = a
c = a.copy()
print('Before:', a == c, a is c, a is b)

b.append(30)
print('After mutation:', a, b, c, a is b)

b = [99]
print('After reassignment:', a, b, a is b)

Before: True False True
After mutation: [10, 20, 30] [10, 20, 30] [10, 20] True
After reassignment: [10, 20, 30] [99] False


In [16]:
assert a == [10, 20, 30] and c == [10, 20]

In [6]:
type(a) is type(c), a is c, a == c

(True, False, True)

<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** Before: True False True. Mutation: a and b both show [10, 20, 30], while c is [10, 20]. Reassignment changes what b names; a stays [10, 20, 30]. == compares values; is compares identity.

**Teaching note:** Use the phrase “two names, one object.” A Python name binding is a logical relationship. Do not claim id() is a portable physical RAM address. Connect shared references to surprising changes in research code.

</details>

## D03 · A shallow copy keeps nested references

Core example · about 4 minutes · slide 8

**Predict before running:** Does copying the outer dictionary isolate the inner list?

In [23]:
from copy import deepcopy
original = {'prices': [100.0, 101.0]}
shallow = original.copy()
print(original is shallow)
print(original['prices'] is shallow['prices'])
print('Original:', original['prices'])
print('Shallow:', shallow['prices'])
independent = deepcopy(original)
shallow['prices'].append(102.0)
print('Original:', original['prices'])
print('Deep copy:', independent['prices'])
print('Shared inner list:', shallow['prices'] is original['prices'])
print('Independent inner list:', independent['prices'] is original['prices'])
assert independent['prices'] == [100.0, 101.0]

False
True
Original: [100.0, 101.0]
Shallow: [100.0, 101.0]
Original: [100.0, 101.0, 102.0]
Deep copy: [100.0, 101.0]
Shared inner list: True
Independent inner list: False


In [26]:
a = [10, 20]

def mutate(x: list):
    y = x.copy()
    y.append(30)
    return y

print(a)
a = mutate(a)
print(a)

[10, 20]
[10, 20, 30]


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** The original becomes [100.0, 101.0, 102.0]. The deep copy remains [100.0, 101.0]. The inner list is shared by original and shallow.

**Teaching note:** Copy depth follows the structure you need to isolate. Deep copying may cost time and memory and is not a universal fix. Keep this to one nested container.

</details>

## D04 · Mutable defaults

Optional extension · about 3 minutes · slide Appendix / optional

**Predict before running:** Why does the second call contain both tickers?

In [4]:
def add_ticker(ticker, tickers=[]):
    tickers.append(ticker)
    return tickers

print(add_ticker('AAPL'))
print(add_ticker('MSFT'))

def add_ticker_safely(ticker, tickers=None):
    if tickers is None:
        tickers = []
    tickers.append(ticker)
    return tickers

print(add_ticker_safely('AAPL'))
print(add_ticker_safely('MSFT'))

['AAPL']
['AAPL', 'MSFT']
['AAPL']
['MSFT']


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** The default list is created once when def executes, so the calls share it. The corrected version creates a fresh list when the argument is omitted. If a caller explicitly passes a list, the corrected function still mutates that supplied list.

**Teaching note:** Optional interview extension. Freshly run this whole cell when demonstrating so the intentionally shared default starts empty.

</details>

# 2. Functions and decorators

In [32]:
positions = [('aapl', 10, 100), ('nvda', 2, 150)]
print(positions)

def dollar_value(position):
    ticker, shares, price = position
    print('Calulating dollar value for ', ticker)
    return shares * price

positions = sorted(positions, key=dollar_value)
print(positions)

[('aapl', 10, 100), ('nvda', 2, 150)]
Calulating dollar value for  aapl
Calulating dollar value for  nvda
[('nvda', 2, 150), ('aapl', 10, 100)]


## D05 · A calculation with explicit inputs

Core example · about 5 minutes · slide 12

**Predict before running:** What is the value? What should happen if one price is missing?

In [5]:
from math import isclose

def portfolio_value(positions, prices):
    missing = set(positions) - set(prices)
    if missing:
        raise ValueError(f'Missing prices: {sorted(missing)}')
    return sum(shares * prices[ticker]
               for ticker, shares in positions.items())

holdings = {'AAPL': 2, 'MSFT': 1}
prices = {'AAPL': 100.0, 'MSFT': 200.0}
print('Value:', portfolio_value(holdings, prices))
assert isclose(portfolio_value(holdings, prices), 400.0)
assert holdings == {'AAPL': 2, 'MSFT': 1}
try:
    portfolio_value(holdings, {'AAPL': 100.0})
except ValueError as error:
    print('Expected failure:', error)

Value: 400.0
Expected failure: Missing prices: ['MSFT']


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** 400.0. A missing MSFT price raises a specific ValueError. The calculation returns a value without modifying its input dictionaries. The instructor supplies the known-answer check independently of the implementation.

**Teaching note:** Name the contract before the code: holdings and matching prices in one currency; value out; missing coverage is an error. This small example checks coverage, not every production concern such as timestamps or currency conversion.

</details>

## D06 · Functions can be inputs

Core example · about 3 minutes · slide 13

**Predict before running:** Which position appears first when sorting by dollar value?

In [6]:
positions = [('AAPL', 2, 100.0), ('MSFT', 1, 250.0)]

def dollar_value(position):
    ticker, shares, price = position
    return shares * price

ordered = sorted(positions, key=dollar_value, reverse=True)
print(ordered)
print(sorted(positions, key=lambda p: p[1] * p[2], reverse=True))
assert ordered[0][0] == 'MSFT'

[('MSFT', 1, 250.0), ('AAPL', 2, 100.0)]
[('MSFT', 1, 250.0), ('AAPL', 2, 100.0)]


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** MSFT appears first: 250 is greater than 200. sorted receives the function itself and calls it for each item. The lambda expresses the same single calculation.

**Teaching note:** Contrast key=dollar_value with calling dollar_value immediately. Keep map/filter/reduce as reference vocabulary rather than adding a syntax tour.

</details>

## D07 · A timing decorator

Core example · about 4 minutes · slide 15

**Predict before running:** Does the decorated function still return 400? Which code handles timing?

In [41]:
from functools import wraps
from time import perf_counter

def timed(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        print(*args)
        print(**kwargs)
        print(function)
        result = function(*args, **kwargs)
        elapsed_ms = (perf_counter() - start) * 1000
        print(f'{function.__name__}: {elapsed_ms:.3f} ms')
        return result
    return wrapper

@timed
def position_value(shares, price):
    return shares * price

@timed
def position_size(shares):
    return shares

print('Returned:', position_value(2, 200.0))
assert position_value.__name__ == 'position_value'

print('Returned:', position_size(2))
assert position_size.__name__ == 'position_size'

2 200.0

<function position_value at 0x11c7620c0>
position_value: 0.214 ms
Returned: 400.0
2

<function position_size at 0x11c761620>
position_size: 0.009 ms
Returned: 2


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** The return value is 400.0. The wrapper measures elapsed time and returns the original result. @timed is equivalent to rebinding position_value = timed(position_value). wraps preserves useful metadata. Exact elapsed time varies.

**Teaching note:** The goal is reading this pattern, not memorizing nested syntax. This simple wrapper reports successful calls; it does not report timing if the function raises. Logging itself is a side effect.

</details>

# 3. Objects and composition

## D08 · Classes and independent instance state

Core example · about 6 minutes · slide 17

**Predict before running:** If we change first.shares, does second change? What does self refer to?

In [49]:
class Position:
    asset_type = 'equity'

    def __init__(self, ticker, shares):
        self.ticker = ticker
        self.shares = shares

    def market_value(self, price):
        return self.shares * price

first = Position('AAPL', 2)
second = Position('AAPL', 2) 
first.shares = 3
print('Values:', first.market_value(100), second.market_value(100))
print('Class attribute:', Position.asset_type)
assert first.shares == 3 and second.shares == 2

Values: 300 200
Class attribute: equity


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** The values are 300 and 200. first and second hold distinct instance state. self names the instance receiving the method call. asset_type is a class attribute.

**Teaching note:** Trace construction, state, and one method call. Equivalent call: Position.market_value(first, 100). __init__ initializes a newly created instance; avoid claiming it alone allocates the object.

</details>

## D09 · Shared class attributes

Core example · about 4 minutes · slide 18

**Predict before running:** Why does the second watchlist see the first ticker?

In [52]:
class SharedWatchlist:
    tickers = []

left = SharedWatchlist()
right = SharedWatchlist()
middle = SharedWatchlist()
left.tickers.append('AAPL')
print('Shared:', right.tickers, middle.tickers)

class Watchlist:
    def __init__(self):
        self.tickers = []

left = Watchlist()
right = Watchlist()
left.tickers.append('AAPL')
print('Independent:', right.tickers)
assert right.tickers == []

Shared: ['AAPL'] ['AAPL']
Independent: []


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** The first list belongs to the class and both instances find that same list. The second version assigns a new list to each instance during initialization.

**Teaching note:** Connect this directly to D02 rather than treating it as a separate trick. Mutation acts on the shared object; assignment to an instance attribute is a different operation.

</details>

## D10 · Composition: a portfolio has positions

Core example · about 6 minutes · slide 19

**Predict before running:** Which object knows how to value one position? Which object combines the results?

In [69]:
class Position:
    def __init__(self, ticker, shares):
        self.ticker = ticker
        self.shares = shares

    def market_value(self, price):
        return self.shares * price

class Portfolio:
    def __init__(self, positions):
        self.positions = list(positions)

    def market_value(self, prices):
        return sum(p.market_value(prices[p.ticker])
                   for p in self.positions)

    def __len__(self):
        return len(self.positions)

    def __repr__(self):
        port_name = ''
        for pos in self.positions:
            port_name += pos.ticker + ' '
        return port_name

book = Portfolio([Position('AAPL', 2), Position('MSFT', 1)])
len(book)
print(book)

# print('Positions:', len(book))
# print('Value:', book.market_value({'AAPL': 100, 'MSFT': 200}))
# assert len(book) == 2

AAPL MSFT 


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** Position values itself at a supplied price. Portfolio holds positions and aggregates their values. len(book) returns 2 via __len__. This is a has-a relationship.

**Teaching note:** The list constructor copies the outer container, not the Position objects. That is a deliberate return to reference semantics. A class is useful when related state and behavior belong together; a simple stateless calculation can remain a function.

</details>

## D11 · A small inheritance example

Optional extension · about 3 minutes · slide Appendix / optional

**Predict before running:** What behavior does the subclass reuse and what does it add?

In [56]:
class Position:
    def __init__(self, ticker, shares):
        self.ticker = ticker
        self.shares = shares

    def market_value(self, price):
        return self.shares * price

class TaggedPosition(Position):
    def __init__(self, ticker, shares, strategy):
        super().__init__(ticker, shares)
        self.strategy = strategy

position = TaggedPosition('AAPL', 2, 'value')
print(position.market_value(100), position.strategy)
print(isinstance(position, TaggedPosition))
print(isinstance(position, Position))
# assert isinstance(position, Position)

200 value
True
True


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** The subclass reuses market_value and initializes the inherited state through super(). It adds a strategy tag. A tag alone may not justify a subclass in a real design; an extra field could be simpler.

**Teaching note:** Use this to teach reading inheritance, then ask whether the subclass is necessary. Keep multiple inheritance and method-resolution-order details in optional study.

</details>

# 4. Iteration and memory use

## D12 · An iterator remembers progress

Core example · about 4 minutes · slide 24

**Predict before running:** What happens when we consume the iterator a second time?

In [12]:
prices = [100, 102, 101]
cursor = iter(prices)
print('First:', next(cursor))
print('Remaining:', list(cursor))
print('Again:', list(cursor))
print('Fresh iterator:', list(iter(prices)))
assert list(cursor) == []

First: 100
Remaining: [102, 101]
Again: []
Fresh iterator: [100, 102, 101]


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** First: 100. Remaining: [102, 101]. Again: []. A fresh iterator over the list gives all three values. The iterable can produce an iterator; the iterator tracks its own progress.

**Teaching note:** Explain StopIteration briefly without leaving a failing cell in the notebook. A list is iterable but not itself an iterator.

</details>

## D13 · A generator yields one result at a time

Core example · about 5 minutes · slide 25

**Predict before running:** Does creating the generator execute its body? What remains after one next call?

In [13]:
def daily_returns(prices):
    for previous, current in zip(prices, prices[1:]):
        yield current / previous - 1

stream = daily_returns([100.0, 110.0, 99.0])
print('First return:', round(next(stream), 3))
print('Remaining:', [round(x, 3) for x in stream])
print('Exhausted:', list(stream))
assert list(stream) == []

First return: 0.1
Remaining: [-0.1]
Exhausted: []


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** The body starts when iteration begins. It yields 0.1, then -0.1, then finishes. yield suspends the function and retains its execution state.

**Teaching note:** This generator makes its outputs lazy, but the input list remains in memory and prices[1:] creates a slice. Do not present it as fully streaming from disk. D14 supplies a cleaner memory comparison.

</details>

## D14 · Materializing versus streaming

Core example · about 5 minutes · slide 26

**Predict before running:** Both calculations visit N values. Which needs to retain all of them?

In [14]:
from time import perf_counter
import gc
import tracemalloc

N = 100_000

def measure(function):
    gc.collect()
    tracemalloc.start()
    start = perf_counter()
    result = function()
    elapsed_ms = (perf_counter() - start) * 1000
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, peak_bytes / (1024 ** 2), elapsed_ms

materialized = measure(lambda: sum([i * i for i in range(N)]))
streamed = measure(lambda: sum(i * i for i in range(N)))
for label, (result, peak_mib, elapsed_ms) in [
    ('list', materialized), ('generator', streamed)
]:
    print(f'{label:9} peak={peak_mib:.3f} MiB  time={elapsed_ms:.1f} ms')
print('Equal answers:', materialized[0] == streamed[0])
assert materialized[0] == streamed[0] == N * (N - 1) * (2 * N - 1) // 6

list      peak=3.815 MiB  time=39.6 ms
generator peak=0.001 MiB  time=38.5 ms
Equal answers: True


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** The results agree. The list retains all squared values; the generator supplies them incrementally. Both do O(N) arithmetic. Streaming avoids the O(N) retained collection, but is not guaranteed to run faster.

**Teaching note:** Measurements are Python allocations observed by tracemalloc, not total process RAM. Instrumentation affects timing. Do not read a universal speed ratio from this tiny example; explain the memory trend.

</details>

## D15 · Representation changes the memory budget

Core example · about 5 minutes · slide 27

**Predict before running:** How much element storage do 100,000 float64 values need? What changes with float32?

In [15]:
import numpy as np
import sys

N = 100_000
values = [float(i) for i in range(N)]
wide = np.array(values, dtype=np.float64)
narrow = wide.astype(np.float32)
list_estimate = sys.getsizeof(values) + sum(sys.getsizeof(x) for x in values)
print(f'List + distinct floats, approximate: {list_estimate:,} bytes')
print(f'float64 element buffer: {wide.nbytes:,} bytes')
print(f'float32 element buffer: {narrow.nbytes:,} bytes')
print('0.1 represented as float32:', format(float(np.float32(0.1)), '.17g'))
assert wide.nbytes == N * 8 and narrow.nbytes == N * 4

List + distinct floats, approximate: 3,200,984 bytes
float64 element buffer: 800,000 bytes
float32 element buffer: 400,000 bytes
0.1 represented as float32: 0.10000000149011612


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** float64 uses 800,000 element-buffer bytes and float32 uses 400,000. The list estimate includes its references and the distinct float objects created here. NumPy nbytes excludes array-object metadata. Lower precision changes representable values and rounding.

**Teaching note:** Compare the units and accounting boundaries, not only the totals. Numeric NumPy arrays store fixed-width elements contiguously in this example. Do not claim all arrays are contiguous or all lower precision is acceptable in financial calculations.

</details>

# 5. Caching and execution

## D16 · Caching reuses work

Core example · about 4 minutes · slide 33

**Predict before running:** How many times does the function body run? What happens when its hidden rate changes?

In [16]:
from functools import lru_cache

rate = 0.05
executions = 0

@lru_cache(maxsize=16)
def discounted_value(amount, years):
    global executions
    executions += 1
    return amount / (1 + rate) ** years

print(round(discounted_value(110, 2), 2))
print(round(discounted_value(110, 2), 2))
rate = 0.10
print('After hidden rate change:', round(discounted_value(110, 2), 2))
print('Body executions:', executions)
print('Cache:', discounted_value.cache_info())

@lru_cache(maxsize=16)
def explicit_discounted_value(amount, years, rate):
    return amount / (1 + rate) ** years

print('Explicit new rate:', round(explicit_discounted_value(110, 2, 0.10), 2))
assert executions == 1
assert round(explicit_discounted_value(110, 2, 0.10), 2) == 90.91

99.77
99.77
After hidden rate change: 99.77
Body executions: 1
Cache: CacheInfo(hits=2, misses=1, maxsize=16, currsize=1)
Explicit new rate: 90.91


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** The body executes once. All three initial calls return about 99.77 because the cache key contains amount and years, not the global rate. Passing the rate explicitly makes the changed dependency part of the key; the corrected result is 90.91.

**Teaching note:** Cache hits trade retained memory for avoided work. Cached arguments must be hashable. This example deliberately combines a hidden global and caching to make stale results visible. A CPU cache and a KV cache store different information.

</details>

## D17 · A Python loop and a NumPy operation

Optional extension · about 3 minutes · slide Appendix / optional

**Predict before running:** Will NumPy use a GPU here? Does faster arithmetic prove the whole program is faster?

In [17]:
import numpy as np
from time import perf_counter
from statistics import median

values = [float(i) for i in range(100_000)]
array = np.array(values, dtype=np.float64)

def loop_total():
    return sum(x * x for x in values)

def array_total():
    return float(np.sum(array * array))

def median_ms(function):
    samples = []
    for _ in range(5):
        start = perf_counter()
        function()
        samples.append((perf_counter() - start) * 1000)
    return median(samples)

assert np.isclose(loop_total(), array_total())
print('Loop ms:', round(median_ms(loop_total), 3))
print('NumPy ms:', round(median_ms(array_total), 3))
print('Array construction is outside the timed region.')

Loop ms: 2.993
NumPy ms: 0.03
Array construction is outside the timed region.


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** This ordinary NumPy example runs on the CPU. Its numerical work executes in compiled routines, reducing Python-level overhead. Array construction is excluded, and the array multiplication creates a temporary. Exact speed ratios depend on the system.

**Teaching note:** Optional. Useful if students assume NumPy automatically means GPU or parallelism. Compare equivalent outputs before timings; state precisely what the timed region includes.

</details>

# 6. Agent tools and independent checks

## D18 · A Python function as an agent tool

Core example · about 6 minutes · slide 37

**Predict before running:** Which component proposes the arguments? Which component actually runs Python? What happens to invalid input?

In [18]:
import math

# Long-only, single-currency classroom example. Type hints are not validation.
def position_value(shares: float, price: float) -> float:
    for name, value in [('shares', shares), ('price', price)]:
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise ValueError(f'{name} must be a number')
        if not math.isfinite(value) or value < 0:
            raise ValueError(f'{name} must be finite and nonnegative')
    result = shares * price
    if not math.isfinite(result):
        raise ValueError('position value overflow')
    return result

# Illustrative request, authored for class. No model or network call occurs.
request = {'tool': 'position_value', 'arguments': {'shares': 2, 'price': 100.0}}
tools = {'position_value': position_value}
value = tools[request['tool']](**request['arguments'])
print({'status': 'ok', 'value': value, 'currency': 'USD'})

for arguments in [{'shares': -2, 'price': 100.0}, {'shares': 2, 'price': float('nan')}]:
    try:
        tools['position_value'](**arguments)
    except ValueError as error:
        print({'status': 'error', 'message': str(error)})
assert value == 200.0

{'status': 'ok', 'value': 200.0, 'currency': 'USD'}
{'status': 'error', 'message': 'shares must be finite and nonnegative'}
{'status': 'error', 'message': 'price must be finite and nonnegative'}


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** In a real agent, a model may propose a tool name and arguments; the harness validates/dispatches and executes the function. Here a prepared dictionary stands in for the request. Valid input returns 200.0. Negative shares and NaN fail under this deliberately long-only contract.

**Teaching note:** Trace request, dispatch, function, result. This is a tiny dispatcher, not a production harness. Validation remains necessary even with type hints. Short positions would require a different contract; negative shares are not universally invalid finance data.

</details>

## D19 · Independent checks catch a plausible defect

Core example · about 6 minutes · slide 38

**Predict before running:** A function returns a number without crashing. What evidence would show that it is wrong?

In [19]:
from math import isclose

# Reference answers are defined independently of either candidate.
CASES = [
    ('ordinary', 2, 100.0, 200.0),
    ('zero holdings', 0, 100.0, 0.0),
    ('fractional', 0.5, 80.0, 40.0),
]

def candidate_bad(shares, price):
    return shares + price

def candidate_fixed(shares, price):
    return shares * price

def grade(function, cases):
    passed = 0
    for name, shares, price, expected in cases:
        actual = function(shares, price)
        ok = isclose(actual, expected, rel_tol=1e-12, abs_tol=1e-12)
        passed += int(ok)
        print(f'{name}: {"PASS" if ok else "FAIL"}; actual={actual}; expected={expected}')
    return passed

print('Candidate A')
a_passed = grade(candidate_bad, CASES)
print('Candidate B')
b_passed = grade(candidate_fixed, CASES)
assert a_passed == 0 and b_passed == 3
print('Development checks passed:', b_passed, '/', len(CASES))

Candidate A
ordinary: FAIL; actual=102.0; expected=200.0
zero holdings: FAIL; actual=100.0; expected=0.0
fractional: FAIL; actual=80.5; expected=40.0
Candidate B
ordinary: PASS; actual=200.0; expected=200.0
zero holdings: PASS; actual=0.0; expected=0.0
fractional: PASS; actual=40.0; expected=40.0
Development checks passed: 3 / 3


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** The faulty addition implementation fails all three checks. Multiplication passes them. The checks assess this arithmetic behavior, not an entire agent’s reliability. The reference answers come from hand-computable cases.

**Teaching note:** Run A before B; ask students to identify the defect from actual versus expected. A self-check instruction tells an agent what to inspect. An independent grader measures the artifact. Since all cases and code are visible here, these are development checks, not held-out evidence.

</details>

## D20 · Transfer question: a new check

Core example · about 3 minutes · slide 42

**Predict before running:** Write the expected result before running: 3 shares at 12.5. Then name one failure mode this case does not test.

In [20]:
from math import isclose

def position_value(shares, price):
    return shares * price

expected = 37.5
actual = position_value(3, 12.5)
print('Transfer check:', actual)
assert isclose(actual, expected)

Transfer check: 37.5


<details>
<summary>Instructor answer and teaching notes</summary>

**Answer:** 37.5. This case does not test input validation, missing prices, incorrect timestamps, currency mismatches, tool selection, or a multi-step workflow. Once students see or tune against it, it is no longer an untouched final evaluation.

**Teaching note:** Use during the closing discussion. Ask for a new case and its expected behavior before looking at implementation.

</details>

# Closing oral questions

1. Why can mutation affect a caller while local reassignment does not?
2. When would you use a class instead of a function?
3. Why can a generator save memory without saving time?
4. How does a Python result cache differ from a CPU cache and a KV cache?
5. What does a passing check establish, and what remains uncertain?

<details><summary>Instructor answer key</summary>

1. The caller and parameter can refer to the same object. Mutation changes that object; local reassignment changes the local binding.
2. A class can group meaningful persistent state with related behavior. A stateless calculation can stay a function. Explain responsibilities rather than adding classes automatically.
3. Generators produce values incrementally, avoiding a retained collection, but still perform the work and may add iteration overhead.
4. A result cache maps arguments to results; CPU caches retain memory data near execution; a KV cache retains earlier tokens’ attention keys and values for reuse during inference.
5. The tested property passed for the tested conditions. Other inputs, provenance, numerical error, resource use, and whole-agent behavior may remain untested.

</details>

No new assignment or deadline is introduced in this notebook.

## Reference reading

- [Python classes, scopes, iterators and generators](https://docs.python.org/3.12/tutorial/classes.html)
- [Python copy](https://docs.python.org/3.12/library/copy.html)
- [Python functools](https://docs.python.org/3.12/library/functools.html)
- [Python tracemalloc](https://docs.python.org/3.12/library/tracemalloc.html)
- [NumPy array storage](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.nbytes.html)
- [NVIDIA CPU/GPU programming model](https://docs.nvidia.com/cuda/archive/13.1.0/cuda-programming-guide/01-introduction/programming-model.html)
- [Hugging Face attention caching](https://huggingface.co/docs/transformers/main/cache_explanation)